# درس ۱۳ - حافظهٔ عامل


## راه‌اندازی

این نوت‌بوک نحوه ساخت یک عامل رزرو سفر با **حافظه پایدار** را با استفاده از **چارچوب عامل مایکروسافت** (MAF) نشان می‌دهد.

شما یاد خواهید گرفت که چگونه انواع مختلف حافظه عامل — کاری، کوتاه‌مدت و بلندمدت — بر نحوه حفظ و استفاده عامل از اطلاعات در طول مکالمات تأثیر می‌گذارد.

**پیش‌نیازها:**
- یک پروژه Microsoft Foundry با یک مدل گپ مستقر شده (مثلاً `gpt-4.1-mini`).
- ورود به سیستم با Azure CLI — در ترمینال خود دستور `az login` را اجرا کنید.
- `AZURE_AI_PROJECT_ENDPOINT` — نقطه انتهایی پروژه Microsoft Foundry شما.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## انواع حافظه عامل

عوامل هوش مصنوعی می‌توانند از انواع مختلف حافظه استفاده کنند، که هر کدام هدف متفاوتی را دنبال می‌کنند:

### حافظه کاری
خود رشته مکالمه — پیام‌های رد و بدل شده در یک جلسه واحد. عامل می‌تواند برای حفظ انسجام به پیام‌های قبلی در همان رشته مراجعه کند. در MAF این حافظه از طریق **`agent.create_session()`** ایجاد می‌شود که یک `AgentSession` برمی‌گرداند.

### حافظه کوتاه‌مدت
اطلاعاتی که در مدت زمان انجام یک کار یا جلسه حفظ می‌شود اما به صورت دائمی ذخیره نمی‌شود. به عنوان مثال، عامل ممکن است در طول یک مکالمه برنامه‌ریزی چند مرحله‌ای حقایقی را جمع‌آوری کرده و از آن‌ها برای تولید یک برنامه نهایی استفاده کند.

### حافظه بلندمدت
ترجیحات و حقایقی که **در طول جلسات مختلف** پابرجا می‌مانند. یک کاربر بازگشتی نباید مجبور باشد محدودیت‌های غذایی یا سبک سفر خود را دوباره تکرار کند. حافظه بلندمدت معمولاً توسط یک ذخیره‌گاه خارجی — مانند پایگاه داده، فایل یا نمایه وکتور — پشتیبانی می‌شود و از طریق ابزارها برای عامل عرضه می‌شود.


## حافظه کاری با جلسات

ساده‌ترین شکل حافظه، جلسه مکالمه است. زمانی که همان جلسه (که از طریق `agent.create_session()` ایجاد شده است) را به تماس‌های متوالی `agent.run()` می‌دهید، عامل تاریخچه کامل آن مکالمه را می‌بیند و می‌تواند جزئیات قبلی را به خاطر بسپارد.

بیایید یک عامل سفر ایجاد کنیم و حافظه کاری را نشان دهیم.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

عامل بودجه را به درستی به خاطر آورد زیرا هر دو پیام در یک جلسه مشترک هستند. این همان **حافظه کاری** است — که فقط برای مدت زمان جلسه وجود دارد.

### با یک رشته جدید چه اتفاقی می‌افتد؟

اگر ما یک جلسه **جدید** ایجاد کنیم، عامل هیچ حافظه‌ای از مکالمه قبلی ندارد:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## الگوی حافظه بلندمدت

برای به خاطر سپردن ترجیحات کاربر **در طول جلسات مختلف**، به یک مخزن پایدار نیاز داریم که خارج از رشته گفتگو زنده بماند. عامل از طریق **ابزارها** به این مخزن دسترسی پیدا می‌کند — توابعی که می‌تواند برای ذخیره و بازیابی اطلاعات فراخوانی کند.

در ادامه یک مخزن ترجیحات ساده در حافظه پیاده‌سازی می‌کنیم (در محیط تولید این را با یک پایگاه داده یا نمایه برداری پشتیبانی می‌کنید) و آن را به عنوان ابزارهایی که عامل می‌تواند استفاده کند، در دسترس قرار می‌دهیم.

### معماری
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### سناریو ۱ — کاربر برای اولین بار سفر سالگرد را رزرو می‌کند

سارا برای اولین بار مراجعه می‌کند. نماینده باید ترجیحات او را از طریق ابزارها ذخیره کند و از آن‌ها برای پیشنهاد هتل‌ها استفاده نماید.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### سناریو ۲ — سارا هفته‌ها بعد بازمی‌گردد

سارا یک **رشته کاملاً جدید** آغاز می‌کند (شبیه‌سازی یک جلسه جدید). حافظه کاری خالی است، اما مخزن ترجیحات بلندمدت هنوز اطلاعات او را دارد. عامل باید آن را بازیابی کند و برای شخصی‌سازی توصیه‌ها استفاده نماید.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصه

در این درس سه نوع حافظه عامل را یاد گرفتید و نحوه پیاده‌سازی آنها را با چارچوب Microsoft Agent آموختید:

| نوع حافظه | مکانیزم MAF | طول عمر |
|---|---|---|
| **کاری** | `agent.create_session()` | یک مکالمه |
| **کوتاه‌مدت** | زمینه جمع شده درون یک رشته | یک وظیفه / جلسه |
| **بلندمدت** | مخزن خارجی که از طریق توابع `@tool` قابل دسترسی است | در طول چند جلسه |

### نکات کلیدی
1. **`agent.create_session()`** حافظه کاری را فراهم می‌کند — عامل کل تاریخچه مکالمه را در یک جلسه می‌بیند.
2. **جلسات جدید زمینه را از دست می‌دهند** — بدون حافظه بلندمدت عامل نمی‌تواند مکالمات گذشته را به یاد آورد.
3. **توابع `@tool`** پل ارتباطی هستند — آنها اجازه می‌دهند عامل اطلاعات را در یک مخزن پایدار ذخیره و بازیابی کند.
4. **شخصی‌سازی با گذر زمان بهبود می‌یابد** — هرچه ترجیحات بیشتری ذخیره شود، توصیه‌های عامل بهتر خواهد بود.

### کاربردهای واقعی
- **خدمات مشتری**: یادآوری تاریخچه و ترجیحات مشتری
- **دستیارهای شخصی**: حفظ زمینه در طول روزها یا هفته‌ها
- **بهداشت و درمان**: پیگیری اطلاعات و ترجیحات بیمار
- **تجارت الکترونیک**: خرید شخصی‌سازی شده بر اساس تاریخچه

### گام‌های بعدی
- جایگزینی دیکشنری درون حافظه با پایگاه داده یا مخزن برداری (مثلاً Azure AI Search)
- اضافه کردن انقضای حافظه برای اطلاعات حساس به زمان
- ساخت سیستم‌های چندعاملی با حافظه مشترک
- کاوش دفترچه یادداشت Cognee برای حافظه مبتنی بر گراف دانش


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
